# E-Commerce Data Quality Assessment

### Seven-Table Integrity and Completeness Audit


## Dataset and analysis scope

Seven supplied CSV tables are used: `customers`, `orders`, `order_items`,
`payments`, `products`, `sellers`, and `geolocation`. They represent a Brazilian
e-commerce dataset. The exact original source and reuse license should be
verified and credited before public publication.

**Financial metric definitions:** Product sales = sum of `order_items.price`
(excludes freight). Payment value = sum of `payments.payment_value`
(may include freight or other payment components). Neither is net profit.

**Coverage limitation:** 2016 records are partial. In 2018, only 20 orders are
recorded in September–October and none afterward; therefore, year comparisons
use January–August 2017 versus January–August 2018.

**Safety:** These notebooks only read from the already-imported `ecommerce`
database. The separate import script is intentionally guarded against
re-importing existing tables. Keep database credentials out of GitHub.


## 1. Environment Setup

All checks are read-only; no data is changed.


In [ ]:
import os
from getpass import getpass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

# Environment variables are optional; no secrets are stored in the notebook.
db = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST", "localhost"),
    user=os.getenv("MYSQL_USER", "root"),
    password=os.getenv("MYSQL_PASSWORD") or getpass("MySQL password: "),
    database=os.getenv("MYSQL_DATABASE", "ecommerce"),
)
cur = db.cursor()
print("Connected to MySQL database:", db.database)


## 2. Data Quality Assessment

### 2.1 Database Schema Inspection

**Objective:** Examine the structure of the imported MySQL tables, including column names, data types, and nullability.

**Importance:** Understanding the database schema helps identify potential data quality issues before finalizing analytical findings.

### Scope of validation

Checks below are **read-only**. Flags indicate values that may need business
review; they are not proof that records are erroneous. Date strings remain in
the original MySQL `TEXT` columns; conversions were validated separately.
Zero freight may reflect legitimate free shipping. Geolocation outliers are
based on approximate screening boundaries, not an official country polygon.


In [ ]:
# Inspect database table structures

tables = [
    "customers",
    "geolocation",
    "order_items",
    "orders",
    "payments",
    "products",
    "sellers"
]

for table in tables:

    print(f"\n{'=' * 50}")
    print(f"TABLE: {table.upper()}")
    print(f"{'=' * 50}")

    cur.execute(f"DESCRIBE `{table}`")

    columns = cur.fetchall()

    df_schema = pd.DataFrame(
        columns,
        columns=[
            "Column",
            "Data Type",
            "Nullable",
            "Key",
            "Default",
            "Extra"
        ]
    )

    display(df_schema)


TABLE: CUSTOMERS


,Column,Data Type,Nullable,Key,Default,Extra
0,customer_id,text,YES,,None,
1,customer_unique_id,text,YES,,None,
2,customer_zip_code_prefix,bigint,YES,,None,
3,customer_city,text,YES,,None,
4,customer_state,text,YES,,None,



TABLE: GEOLOCATION


,Column,Data Type,Nullable,Key,Default,Extra
0,geolocation_zip_code_prefix,bigint,YES,,None,
1,geolocation_lat,double,YES,,None,
2,geolocation_lng,double,YES,,None,
3,geolocation_city,text,YES,,None,
4,geolocation_state,text,YES,,None,



TABLE: ORDER_ITEMS


,Column,Data Type,Nullable,Key,Default,Extra
0,order_id,text,YES,,None,
1,order_item_id,bigint,YES,,None,
2,product_id,text,YES,,None,
3,seller_id,text,YES,,None,
4,shipping_limit_date,text,YES,,None,
5,price,double,YES,,None,
6,freight_value,double,YES,,None,


... (4 outputs left out)


TABLE: PRODUCTS


,Column,Data Type,Nullable,Key,Default,Extra
0,product_id,text,YES,,None,
1,product_category,text,YES,,None,
2,product_name_length,double,YES,,None,
3,product_description_length,double,YES,,None,
4,product_photos_qty,double,YES,,None,
5,product_weight_g,double,YES,,None,
6,product_length_cm,double,YES,,None,
7,product_height_cm,double,YES,,None,
8,product_width_cm,double,YES,,None,



TABLE: SELLERS


,Column,Data Type,Nullable,Key,Default,Extra
0,seller_id,text,YES,,None,
1,seller_zip_code_prefix,bigint,YES,,None,
2,seller_city,text,YES,,None,
3,seller_state,text,YES,,None,


### 2.2 Missing Value Analysis — Customers

**Objective:** Identify missing values in important customer attributes.

**Importance:** Missing customer identifiers or geographical information may affect customer segmentation, retention analysis, and location-based reporting.

In [ ]:
# Check missing values in customers table

query = """
SELECT
    COUNT(*) AS total_records,

    SUM(customer_id IS NULL) AS missing_customer_id,

    SUM(customer_unique_id IS NULL) AS missing_unique_id,

    SUM(customer_zip_code_prefix IS NULL) AS missing_zip,

    SUM(customer_city IS NULL) AS missing_city,

    SUM(customer_state IS NULL) AS missing_state

FROM customers;
"""

cur.execute(query)

data = cur.fetchall()

df_missing_customers = pd.DataFrame(
    data,
    columns=[
        "Total Records",
        "Missing Customer IDs",
        "Missing Unique IDs",
        "Missing ZIP Codes",
        "Missing Cities",
        "Missing States"
    ]
)

display(df_missing_customers)

,Total Records,Missing Customer IDs,Missing Unique IDs,Missing ZIP Codes,Missing Cities,Missing States
0,99441,0,0,0,0,0


**Results:**

- Total Customer Records: 99,441
- Missing Customer IDs: 0
- Missing Unique Customer IDs: 0
- Missing ZIP Codes: 0
- Missing Cities: 0
- Missing States: 0

**Data Quality Observation:**

No SQL NULL values were identified in the five customer attributes examined.

This is a positive initial data quality finding. However, further validation is required to identify duplicate identifiers, empty strings, and potential formatting inconsistencies.

**Next Step:** Validate customer identifier uniqueness and check for duplicate records.

### 2.3 Duplicate Customer Identifier Analysis

**Objective:** Verify whether customer_id values are unique across customer records.

**Importance:** Duplicate customer identifiers may cause incorrect JOIN results and distort customer-level analysis.

**Note:** customer_unique_id is expected to repeat when the same individual places multiple orders.

In [ ]:
# Check duplicate customer identifiers

query = """
SELECT
    COUNT(*) AS total_records,

    COUNT(DISTINCT customer_id) AS unique_customer_ids,

    COUNT(*) - COUNT(DISTINCT customer_id)
        AS duplicate_customer_ids,

    COUNT(DISTINCT customer_unique_id)
        AS unique_individual_customers,

    SUM(
        CASE
            WHEN TRIM(customer_id) = '' THEN 1
            ELSE 0
        END
    ) AS blank_customer_ids

FROM customers;
"""

cur.execute(query)

data = cur.fetchall()

df_duplicates = pd.DataFrame(
    data,
    columns=[
        "Total Records",
        "Unique Customer IDs",
        "Duplicate Customer IDs",
        "Unique Individual Customers",
        "Blank Customer IDs"
    ]
)

display(df_duplicates)

,Total Records,Unique Customer IDs,Duplicate Customer IDs,Unique Individual Customers,Blank Customer IDs
0,99441,99441,0,96096,0


**Results:**

- Total Customer Records: 99,441
- Unique Customer IDs: 99,441
- Duplicate Customer IDs: 0
- Unique Individual Customers: 96,096
- Blank Customer IDs: 0

**Data Quality Findings:**

- All customer_id values are unique.
- No duplicate or blank customer_id values were detected.
- The dataset contains 96,096 distinct individual customers.
- The difference between customer records and unique individuals is consistent with the dataset's order-specific customer identifiers.

**Conclusion:**

The customers table passed the initial identifier uniqueness and SQL NULL checks.

No customer records were removed or modified.

**Remaining Checks:**

- Verify postal code formatting.
- Inspect other tables for duplicate records and missing values.
- Review database column types and key constraints.

In [ ]:
# Inspect orders table schema

cur.execute("DESCRIBE orders")

data = cur.fetchall()

df_orders_schema = pd.DataFrame(
    data,
    columns=[
        "Column",
        "Data Type",
        "Nullable",
        "Key",
        "Default",
        "Extra"
    ]
)

display(df_orders_schema)

,Column,Data Type,Nullable,Key,Default,Extra
0,order_id,text,YES,,None,
1,customer_id,text,YES,,None,
2,order_status,text,YES,,None,
3,order_purchase_timestamp,text,YES,,None,
4,order_approved_at,text,YES,,None,
5,order_delivered_carrier_date,text,YES,,None,
6,order_delivered_customer_date,text,YES,,None,
7,order_estimated_delivery_date,text,YES,,None,


### 2.5 Orders Table — Missing Values and Duplicate Analysis

**Objective:** Identify missing values and duplicate order identifiers in the orders table.

**Importance:** Missing order dates and duplicate order IDs may affect order trends, delivery analysis, and customer purchasing metrics.

**Methodology:** SQL aggregation functions are used to calculate total records, unique order IDs, duplicate identifiers, and missing values in important order attributes.

In [ ]:
# Orders Table: Missing Values and Duplicates

query = """
SELECT
    COUNT(*) AS total_orders,

    COUNT(DISTINCT order_id) AS unique_orders,

    COUNT(*) - COUNT(DISTINCT order_id)
        AS duplicate_order_ids,

    SUM(order_id IS NULL)
        AS missing_order_id,

    SUM(customer_id IS NULL)
        AS missing_customer_id,

    SUM(order_status IS NULL)
        AS missing_order_status,

    SUM(order_purchase_timestamp IS NULL)
        AS missing_purchase_date,

    SUM(order_approved_at IS NULL)
        AS missing_approval_date,

    SUM(order_delivered_carrier_date IS NULL)
        AS missing_carrier_date,

    SUM(order_delivered_customer_date IS NULL)
        AS missing_delivery_date,

    SUM(order_estimated_delivery_date IS NULL)
        AS missing_estimated_date

FROM orders;
"""

cur.execute(query)

data = cur.fetchall()

df_orders_quality = pd.DataFrame(
    data,
    columns=[
        "Total Orders",
        "Unique Orders",
        "Duplicate Order IDs",
        "Missing Order IDs",
        "Missing Customer IDs",
        "Missing Order Status",
        "Missing Purchase Dates",
        "Missing Approval Dates",
        "Missing Carrier Dates",
        "Missing Delivery Dates",
        "Missing Estimated Dates"
    ]
)

# Display results vertically for readability
display(df_orders_quality.T.rename(columns={0: "Count"}))

,Count
Total Orders,99441
Unique Orders,99441
Duplicate Order IDs,0
Missing Order IDs,0
Missing Customer IDs,0
Missing Order Status,0
Missing Purchase Dates,0
Missing Approval Dates,160
Missing Carrier Dates,1783
Missing Delivery Dates,2965


**Results:**

- Total Orders: 99,441
- Unique Order IDs: 99,441
- Duplicate Order IDs: 0
- Missing Order IDs: 0
- Missing Customer IDs: 0
- Missing Order Status: 0
- Missing Purchase Dates: 0
- Missing Approval Dates: 160 (0.16%)
- Missing Carrier Dates: 1,783 (1.79%)
- Missing Customer Delivery Dates: 2,965 (2.98%)
- Missing Estimated Delivery Dates: 0

**Data Quality Findings:**

- No duplicate order identifiers were detected.
- All orders contain customer IDs, order statuses, purchase timestamps, and estimated delivery dates.
- Missing values are concentrated in approval and delivery-related date fields.
- Approximately 2.98% of orders have no recorded customer delivery date.

**Next Investigation:**

Determine whether missing delivery dates are associated with order statuses such as canceled, unavailable, shipped, or processing.

**Data Cleaning Decision:**

Missing delivery dates will not be replaced automatically because their absence may reflect the actual order lifecycle.

### 2.6 Order Status and Missing Delivery Date Analysis

**Objective:** Investigate the relationship between order status and missing delivery dates.

**Importance:** Missing delivery dates may be valid for orders that have not been delivered. Examining missing values by order status helps distinguish expected missing data from potential data quality issues.

**Methodology:** Group orders by status and calculate total orders, missing delivery dates, and the percentage of missing delivery dates within each status.

In [ ]:
# Analyze missing delivery dates by order status

query = """
SELECT
    order_status,

    COUNT(*) AS total_orders,

    SUM(
        order_delivered_customer_date IS NULL
    ) AS missing_delivery_dates,

    ROUND(
        SUM(order_delivered_customer_date IS NULL)
        * 100.0 / NULLIF(COUNT(*), 0),
        2
    ) AS missing_percentage

FROM orders

GROUP BY order_status

ORDER BY total_orders DESC;
"""

cur.execute(query)

data = cur.fetchall()

df_order_status = pd.DataFrame(
    data,
    columns=[
        "Order Status",
        "Total Orders",
        "Missing Delivery Dates",
        "Missing Percentage"
    ]
)

display(df_order_status)

,Order Status,Total Orders,Missing Delivery Dates,Missing Percentage
0,delivered,96478,8,0.01
1,shipped,1107,1107,100.00
2,canceled,625,619,99.04
3,unavailable,609,609,100.00
4,invoiced,314,314,100.00
5,processing,301,301,100.00
6,created,5,5,100.00
7,approved,2,2,100.00


**Results:**

- Delivered Orders: 96,478
- Shipped Orders: 1,107
- Canceled Orders: 625
- Unavailable Orders: 609
- Invoiced Orders: 314
- Processing Orders: 301

**Data Quality Findings:**

- Most missing delivery dates are associated with orders that were not marked as delivered.
- Only 8 of the 96,478 delivered orders have missing customer delivery dates.
- Six canceled orders contain recorded customer delivery dates.
- These unusual combinations require further investigation before being classified as data errors.

**Data Cleaning Decision:**

- Preserve missing delivery dates for orders without confirmed delivery.
- Flag delivered orders with missing delivery timestamps for further investigation.
- Investigate canceled orders that contain actual delivery timestamps.
- Avoid automatically deleting records or replacing missing dates with fabricated values.

**Conclusion:**

Missing delivery dates are largely explained by order statuses. A small number of potentially inconsistent records require additional validation.

### 2.7 Investigation of Potential Order Status Anomalies

**Objective:** Identify delivered orders with missing delivery dates and canceled orders with recorded delivery dates.

**Importance:** Investigating unexpected combinations of order status and delivery timestamps helps improve data reliability without removing potentially valid records.

**Methodology:** Apply conditional SQL filters to identify records requiring manual review.

In [ ]:
# Investigate unusual order status and delivery date combinations

query = """
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date

FROM orders

WHERE
    (
        order_status = 'delivered'
        AND order_delivered_customer_date IS NULL
    )

    OR

    (
        order_status = 'canceled'
        AND order_delivered_customer_date IS NOT NULL
    )

ORDER BY order_status, order_purchase_timestamp;
"""

cur.execute(query)

data = cur.fetchall()

df_anomalies = pd.DataFrame(
    data,
    columns=[
        "Order ID",
        "Customer ID",
        "Order Status",
        "Purchase Date",
        "Actual Delivery Date",
        "Estimated Delivery Date"
    ]
)

display(df_anomalies)

print("Total Potential Anomalies:", len(df_anomalies))

,Order ID,Customer ID,Order Status,Purchase Date,Actual Delivery Date,Estimated Delivery Date
0,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-11-08 10:58:34,2016-11-25 00:00:00
1,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-14 15:07:11,2016-11-29 00:00:00
2,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-19 18:47:43,2016-11-30 00:00:00
3,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-16 14:36:59,2016-11-30 00:00:00
4,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-11-09 14:53:50,2016-12-08 00:00:00
5,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-03-21 22:03:51,2018-03-09 00:00:00
6,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,None,2017-06-23 00:00:00
7,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,None,2017-12-18 00:00:00
8,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,None,2018-06-26 00:00:00
9,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,None,2018-07-16 00:00:00


Total Potential Anomalies: 14


**Results:**

A total of 14 potentially inconsistent order records were identified:

- 8 delivered orders with missing actual delivery timestamps.
- 6 canceled orders with recorded actual delivery timestamps.

**Data Quality Findings:**

- Some delivered orders lack recorded delivery timestamps.
- A small number of canceled orders contain delivery timestamps.
- Five canceled orders with recorded delivery timestamps occurred in October 2016.
- These combinations may reflect operational processes, historical updates, or data recording inconsistencies.

**Data Cleaning Decision:**

- Preserve all 14 records in the original dataset.
- Flag these records as potential anomalies.
- Avoid automatically imputing missing delivery timestamps.
- Exclude records with missing actual delivery dates from analyses requiring a confirmed delivery timestamp.

**Conclusion:**

The orders table contains a small number of potentially inconsistent status-date combinations. These cases have been documented for further investigation without modifying the original records.

### 2.8 Order Date Format Validation

**Objective:** Validate the format and convertibility of date-related columns in the orders table.

**Importance:** Accurate datetime formatting is necessary for time-series analysis, delivery performance calculations, and customer purchasing trends.

**Methodology:** Retrieve date columns from MySQL and use Pandas datetime conversion with error handling to identify invalid date values.

**Data Preservation:** This validation does not modify the original MySQL database.

In [ ]:
# Validate order date formats

query = """
SELECT
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM orders;
"""

cur.execute(query)

data = cur.fetchall()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

df_dates = pd.DataFrame(
    data,
    columns=date_columns
)

validation_results = []

for col in date_columns:

    # Count original non-null values
    original_non_null = df_dates[col].notna().sum()

    # Convert to datetime
    converted = pd.to_datetime(
        df_dates[col],
        errors="coerce"
    )

    # Count valid datetime values
    valid_dates = converted.notna().sum()

    # Calculate invalid formats
    invalid_dates = original_non_null - valid_dates

    # Count missing values
    missing_values = df_dates[col].isna().sum()

    validation_results.append({
        "Column": col,
        "Valid Dates": valid_dates,
        "Invalid Dates": invalid_dates,
        "Missing Values": missing_values
    })

df_date_validation = pd.DataFrame(validation_results)

display(df_date_validation)

,Column,Valid Dates,Invalid Dates,Missing Values
0,order_purchase_timestamp,99441,0,0
1,order_approved_at,99281,0,160
2,order_delivered_carrier_date,97658,0,1783
3,order_delivered_customer_date,96476,0,2965
4,order_estimated_delivery_date,99441,0,0


**Results:**

All non-null order date values were successfully converted to datetime format using Pandas.

- Invalid Purchase Dates: 0
- Invalid Approval Dates: 0
- Invalid Carrier Dates: 0
- Invalid Customer Delivery Dates: 0
- Invalid Estimated Delivery Dates: 0

**Data Quality Findings:**

- No invalid datetime strings were detected in the available date values.
- Missing dates are concentrated in approval and delivery-related fields.
- The original MySQL columns were imported as TEXT, but their non-null values can be interpreted as valid datetimes.

**Data Cleaning Decision:**

- Preserve the original date values in the raw database.
- Convert date columns to appropriate datetime types during the reproducible cleaning process.
- Maintain missing dates as NULL where they represent unavailable information.

**Conclusion:**

The orders table passed datetime format validation. No invalid non-null date values were detected.

### 2.9 Payments Table — Data Quality Assessment

**Objective:** Examine payment records for missing values, invalid payment amounts, and unusual installment information.

**Importance:** Payment data quality directly affects revenue calculations, customer spending analysis, and financial reporting.

**Methodology:** Inspect the database schema and evaluate payment amounts, payment types, and installment counts.

In [ ]:
# Inspect payments table schema

cur.execute("DESCRIBE payments")

data = cur.fetchall()

df_payments_schema = pd.DataFrame(
    data,
    columns=[
        "Column",
        "Data Type",
        "Nullable",
        "Key",
        "Default",
        "Extra"
    ]
)

display(df_payments_schema)

,Column,Data Type,Nullable,Key,Default,Extra
0,order_id,text,YES,,None,
1,payment_sequential,bigint,YES,,None,
2,payment_type,text,YES,,None,
3,payment_installments,bigint,YES,,None,
4,payment_value,double,YES,,None,


### 2.10 Payments Table — Missing Values and Financial Validation

**Objective:** Identify missing payment information, invalid payment amounts, and unusual installment values.

**Importance:** Financial data quality directly affects revenue analysis, payment behavior insights, and customer spending calculations.

**Methodology:** SQL aggregation functions are used to evaluate missing values, zero and negative payment amounts, installment counts, and payment record distribution.

In [ ]:
# Payments Table: Data Quality Validation

query = """
SELECT
    COUNT(*) AS total_payment_records,

    COUNT(DISTINCT order_id) AS unique_orders,

    SUM(order_id IS NULL) AS missing_order_ids,

    SUM(payment_sequential IS NULL) AS missing_payment_sequence,

    SUM(payment_type IS NULL) AS missing_payment_types,

    SUM(payment_installments IS NULL) AS missing_installments,

    SUM(payment_value IS NULL) AS missing_payment_values,

    SUM(payment_value = 0) AS zero_payments,

    SUM(payment_value < 0) AS negative_payments,

    SUM(payment_installments = 0) AS zero_installments,

    SUM(payment_installments < 0) AS negative_installments,

    ROUND(MIN(payment_value), 2) AS minimum_payment,

    ROUND(MAX(payment_value), 2) AS maximum_payment,

    ROUND(AVG(payment_value), 2) AS average_payment

FROM payments;
"""

cur.execute(query)

data = cur.fetchall()

df_payment_quality = pd.DataFrame(
    data,
    columns=[
        "Total Payment Records",
        "Unique Orders",
        "Missing Order IDs",
        "Missing Payment Sequences",
        "Missing Payment Types",
        "Missing Installments",
        "Missing Payment Values",
        "Zero Payments",
        "Negative Payments",
        "Zero Installments",
        "Negative Installments",
        "Minimum Payment",
        "Maximum Payment",
        "Average Payment"
    ]
)

# Display vertically
display(df_payment_quality.T.rename(columns={0: "Value"}))

,Value
Total Payment Records,103886
Unique Orders,99440
Missing Order IDs,0
Missing Payment Sequences,0
Missing Payment Types,0
Missing Installments,0
Missing Payment Values,0
Zero Payments,9
Negative Payments,0
Zero Installments,2


**Results:**

- Total Payment Records: 103,886
- Unique Orders with Payments: 99,440
- Missing Payment Values: 0
- Zero-Value Payments: 9
- Negative Payments: 0
- Zero Installment Records: 2
- Negative Installments: 0
- Minimum Payment Value: BRL 0.00
- Maximum Payment Value: BRL 13,664.08
- Average Payment Record Value: BRL 154.10

**Data Quality Findings:**

- No SQL NULL values were detected in the examined payment columns.
- Multiple payment records can belong to the same order.
- Nine payment records have zero payment value.
- Two payment records contain zero installments.
- No negative payment values or installment counts were detected.

**Next Investigation:**

Examine zero-value payments, zero-installment records, and orders without recorded payments.

**Data Cleaning Decision:**

Preserve all original payment records until their business context has been investigated.

In [ ]:
# Investigate zero payments and zero installments

query = """
SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value

FROM payments

WHERE
    payment_value = 0
    OR payment_installments = 0

ORDER BY
    payment_value,
    payment_type;
"""

cur.execute(query)

data = cur.fetchall()

df_payment_anomalies = pd.DataFrame(
    data,
    columns=[
        "Order ID",
        "Payment Sequence",
        "Payment Type",
        "Installments",
        "Payment Value"
    ]
)

display(df_payment_anomalies)

print("Records Requiring Review:", len(df_payment_anomalies))

,Order ID,Payment Sequence,Payment Type,Installments,Payment Value
0,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
1,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
2,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
3,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
4,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
5,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
6,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
7,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
8,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
9,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


Records Requiring Review: 11


In [ ]:
# Identify orders without payment records

query = """
SELECT
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp

FROM orders AS o

LEFT JOIN payments AS p
    ON o.order_id = p.order_id

WHERE p.order_id IS NULL;
"""

cur.execute(query)

data = cur.fetchall()

df_missing_payments = pd.DataFrame(
    data,
    columns=[
        "Order ID",
        "Order Status",
        "Purchase Date"
    ]
)

display(df_missing_payments)

print("Orders Without Payment Records:", len(df_missing_payments))

,Order ID,Order Status,Purchase Date
0,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38


Orders Without Payment Records: 1


**Results:**

- Total Zero-Value Payment Records: 9
- Zero-Value Payments with Type `not_defined`: 3
- Zero-Value Voucher Payments: 6
- Credit Card Records with Zero Installments: 2
- Orders Without Corresponding Payment Records: 1

**Data Quality Findings:**

- Nine payment records contain a value of zero.
- Six of these records use vouchers, while three have an undefined payment type.
- Two credit card payment records contain zero installments despite having positive payment values.
- One delivered order from September 2016 has no corresponding record in the payments table.

**Business Implications:**

- Unusual payment records may affect payment method and installment analyses.
- Orders without payment records are excluded from calculations based on INNER JOINs with the payments table.
- Unexplained payment values require further investigation before being classified as errors.

**Data Cleaning Decisions:**

- Preserve zero-value payment records.
- Flag zero-installment credit card records for further review.
- Document the delivered order without a matching payment record.
- Avoid imputing payment amounts or installment values without supporting evidence.

**Conclusion:**

The payments table contains a small number of unusual records. These findings have been documented, and no original payment records have been modified.

### 2.12 Order Items Table — Data Quality Assessment

**Objective:** Examine the order items table structure and identify potential issues affecting product-level sales analysis.

**Importance:** Order item quality directly affects product revenue, seller performance, and purchasing frequency calculations.

**Methodology:** Inspect column types before validating item prices, freight values, missing information, and identifier consistency.

In [ ]:
# Inspect order_items table schema

cur.execute("DESCRIBE order_items")

data = cur.fetchall()

df_order_items_schema = pd.DataFrame(
    data,
    columns=[
        "Column",
        "Data Type",
        "Nullable",
        "Key",
        "Default",
        "Extra"
    ]
)

display(df_order_items_schema)

,Column,Data Type,Nullable,Key,Default,Extra
0,order_id,text,YES,,None,
1,order_item_id,bigint,YES,,None,
2,product_id,text,YES,,None,
3,seller_id,text,YES,,None,
4,shipping_limit_date,text,YES,,None,
5,price,double,YES,,None,
6,freight_value,double,YES,,None,


### 2.13 Order Items — Missing Values and Price Validation

**Objective:** Identify missing values, duplicate order-item identifiers, and unusual price or freight values.

**Importance:** Data quality issues in order items may affect category sales, seller rankings, and product purchasing analysis.

**Methodology:** SQL aggregation functions are used to validate missing values, price ranges, freight charges, and item identifier uniqueness.

In [ ]:
# Order Items: Data Quality Validation

query = """
SELECT
    COUNT(*) AS total_order_items,

    COUNT(DISTINCT order_id, order_item_id)
        AS unique_order_item_pairs,

    COUNT(*) - COUNT(DISTINCT order_id, order_item_id)
        AS duplicate_order_item_pairs,

    SUM(order_id IS NULL) AS missing_order_ids,

    SUM(order_item_id IS NULL) AS missing_item_ids,

    SUM(product_id IS NULL) AS missing_product_ids,

    SUM(seller_id IS NULL) AS missing_seller_ids,

    SUM(shipping_limit_date IS NULL) AS missing_shipping_dates,

    SUM(price IS NULL) AS missing_prices,

    SUM(freight_value IS NULL) AS missing_freight_values,

    SUM(price = 0) AS zero_prices,

    SUM(price < 0) AS negative_prices,

    SUM(freight_value = 0) AS zero_freight_values,

    SUM(freight_value < 0) AS negative_freight_values,

    ROUND(MIN(price), 2) AS minimum_price,

    ROUND(MAX(price), 2) AS maximum_price,

    ROUND(AVG(price), 2) AS average_price,

    ROUND(AVG(freight_value), 2) AS average_freight

FROM order_items;
"""

cur.execute(query)

data = cur.fetchall()

df_order_items_quality = pd.DataFrame(
    data,
    columns=[
        "Total Order Items",
        "Unique Order-Item Pairs",
        "Duplicate Order-Item Pairs",
        "Missing Order IDs",
        "Missing Item IDs",
        "Missing Product IDs",
        "Missing Seller IDs",
        "Missing Shipping Dates",
        "Missing Prices",
        "Missing Freight Values",
        "Zero Prices",
        "Negative Prices",
        "Zero Freight Values",
        "Negative Freight Values",
        "Minimum Price",
        "Maximum Price",
        "Average Price",
        "Average Freight"
    ]
)

display(
    df_order_items_quality.T.rename(columns={0: "Value"})
)

,Value
Total Order Items,112650
Unique Order-Item Pairs,112650
Duplicate Order-Item Pairs,0
Missing Order IDs,0
Missing Item IDs,0
Missing Product IDs,0
Missing Seller IDs,0
Missing Shipping Dates,0
Missing Prices,0
Missing Freight Values,0


**Results:**

- Total Order Items: 112,650
- Unique Order-Item Pairs: 112,650
- Duplicate Order-Item Pairs: 0
- Missing Values: 0
- Zero Product Prices: 0
- Negative Product Prices: 0
- Zero Freight Values: 383
- Negative Freight Values: 0
- Minimum Product Price: BRL 0.85
- Maximum Product Price: BRL 6,735.00
- Average Product Price: BRL 120.65
- Average Freight Value: BRL 19.99

**Data Quality Findings:**

- All order-item identifier combinations are unique.
- No SQL NULL values were detected in the examined columns.
- All recorded product prices are positive.
- No negative freight values were identified.
- A total of 383 order items have zero freight charges.
- Product prices exhibit substantial variation.

**Data Cleaning Decisions:**

- Preserve all order-item records.
- Retain zero freight values because they may represent valid business transactions.
- Avoid treating high-priced products as invalid without additional evidence.
- Maintain the original prices for financial analysis.

**Conclusion:**

The order_items table passed the initial missing-value, identifier uniqueness, and numerical validation checks.

Further referential integrity checks are required to confirm that order, product, and seller identifiers match their corresponding tables.

### 2.14 Products Table — Data Quality Assessment

**Objective:** Examine product attributes and identify potential issues affecting product category analysis and product-level reporting.

**Importance:** Product data quality influences category sales analysis, product segmentation, and inventory-related insights.

**Methodology:** Inspect the table schema before checking missing values, duplicate product identifiers, and numerical attributes.

In [ ]:
# Inspect products table schema

cur.execute("DESCRIBE products")

data = cur.fetchall()

df_products_schema = pd.DataFrame(
    data,
    columns=[
        "Column",
        "Data Type",
        "Nullable",
        "Key",
        "Default",
        "Extra"
    ]
)

display(df_products_schema)

,Column,Data Type,Nullable,Key,Default,Extra
0,product_id,text,YES,,None,
1,product_category,text,YES,,None,
2,product_name_length,double,YES,,None,
3,product_description_length,double,YES,,None,
4,product_photos_qty,double,YES,,None,
5,product_weight_g,double,YES,,None,
6,product_length_cm,double,YES,,None,
7,product_height_cm,double,YES,,None,
8,product_width_cm,double,YES,,None,


### 2.15 Products Table — Missing Values and Attribute Validation

**Objective:** Identify missing product attributes, duplicate product identifiers, and invalid physical measurements.

**Importance:** Product data quality affects category-level sales analysis, product segmentation, and logistics-related insights.

**Methodology:** SQL aggregation functions are used to evaluate missing values, product identifier uniqueness, and non-positive physical measurements.

In [ ]:
# Products Table: Data Quality Validation

query = """
SELECT
    COUNT(*) AS total_products,

    COUNT(DISTINCT product_id) AS unique_products,

    COUNT(*) - COUNT(DISTINCT product_id)
        AS duplicate_product_ids,

    SUM(product_id IS NULL) AS missing_product_ids,

    SUM(product_category IS NULL) AS missing_categories,

    SUM(product_name_length IS NULL) AS missing_name_lengths,

    SUM(product_description_length IS NULL)
        AS missing_description_lengths,

    SUM(product_photos_qty IS NULL) AS missing_photo_counts,

    SUM(product_weight_g IS NULL) AS missing_weights,

    SUM(product_length_cm IS NULL) AS missing_lengths,

    SUM(product_height_cm IS NULL) AS missing_heights,

    SUM(product_width_cm IS NULL) AS missing_widths,

    SUM(product_weight_g <= 0) AS non_positive_weights,

    SUM(product_length_cm <= 0) AS non_positive_lengths,

    SUM(product_height_cm <= 0) AS non_positive_heights,

    SUM(product_width_cm <= 0) AS non_positive_widths,

    SUM(product_photos_qty < 0) AS negative_photo_counts

FROM products;
"""

cur.execute(query)

data = cur.fetchall()

df_products_quality = pd.DataFrame(
    data,
    columns=[
        "Total Products",
        "Unique Products",
        "Duplicate Product IDs",
        "Missing Product IDs",
        "Missing Categories",
        "Missing Name Lengths",
        "Missing Description Lengths",
        "Missing Photo Counts",
        "Missing Weights",
        "Missing Lengths",
        "Missing Heights",
        "Missing Widths",
        "Non-Positive Weights",
        "Non-Positive Lengths",
        "Non-Positive Heights",
        "Non-Positive Widths",
        "Negative Photo Counts"
    ]
)

display(
    df_products_quality.T.rename(columns={0: "Value"})
)

,Value
Total Products,32951
Unique Products,32951
Duplicate Product IDs,0
Missing Product IDs,0
Missing Categories,610
Missing Name Lengths,610
Missing Description Lengths,610
Missing Photo Counts,610
Missing Weights,2
Missing Lengths,2


**Results:**

- Total Products: 32,951
- Unique Product IDs: 32,951
- Duplicate Product IDs: 0
- Missing Product Categories: 610 (approximately 1.85%)
- Missing Name Lengths: 610
- Missing Description Lengths: 610
- Missing Photo Counts: 610
- Missing Physical Measurements: 2 per dimension/weight field
- Non-Positive Product Weights: 4
- Non-Positive Product Dimensions: 0
- Negative Photo Counts: 0

**Data Quality Findings:**

- All product identifiers are unique.
- Approximately 1.85% of products have missing category information.
- Product metadata fields contain missing values.
- A small number of products have missing or non-positive weights.
- No non-positive product dimensions or negative photo counts were detected.

**Business Implications:**

- Missing product categories may affect category-level sales analysis.
- Missing product measurements may limit logistics and shipping-related analyses.
- Incomplete product metadata may affect product segmentation and catalog analysis.

**Next Investigation:**

Examine whether the same products have missing category and descriptive attributes, and inspect products with unusual weights.

**Data Cleaning Decision:**

Preserve original records and investigate missing attributes before applying transformations.

### 2.16 Investigation of Missing Product Metadata

**Objective:** Determine whether missing product category and descriptive attributes occur together.

**Importance:** Understanding missing-value patterns helps identify whether data is incomplete at the individual field level or across entire product records.

**Methodology:** SQL conditional aggregation is used to examine overlapping missing attributes.

In [ ]:
# Investigate missing product metadata patterns

query = """
SELECT
    COUNT(*) AS total_products,

    SUM(
        product_category IS NULL
        AND product_name_length IS NULL
        AND product_description_length IS NULL
        AND product_photos_qty IS NULL
    ) AS all_four_attributes_missing,

    SUM(
        product_category IS NULL
        AND (
            product_name_length IS NOT NULL
            OR product_description_length IS NOT NULL
            OR product_photos_qty IS NOT NULL
        )
    ) AS category_missing_other_metadata_present,

    SUM(
        product_weight_g IS NULL
    ) AS missing_weight,

    SUM(
        product_weight_g = 0
    ) AS zero_weight,

    SUM(
        product_weight_g < 0
    ) AS negative_weight

FROM products;
"""

cur.execute(query)

data = cur.fetchall()

df_product_patterns = pd.DataFrame(
    data,
    columns=[
        "Total Products",
        "All Four Attributes Missing",
        "Category Missing but Other Metadata Present",
        "Missing Weight",
        "Zero Weight",
        "Negative Weight"
    ]
)

display(
    df_product_patterns.T.rename(columns={0: "Count"})
)

,Count
Total Products,32951
All Four Attributes Missing,610
Category Missing but Other Metadata Present,0
Missing Weight,2
Zero Weight,4
Negative Weight,0


**Results:**

- Total Products: 32,951
- Products Missing All Four Metadata Attributes: 610
- Products Missing Category but Having Other Metadata: 0
- Missing Product Weights: 2
- Zero Product Weights: 4
- Negative Product Weights: 0

**Data Quality Findings:**

- All 610 products with missing category information also have missing name length, description length, and photo count.
- Missing metadata affects approximately 1.85% of products.
- Two products have missing weight values.
- Four products have zero recorded weight.
- No negative product weights were detected.

**Data Cleaning Decisions:**

- Retain all original product records.
- Represent missing categories as "Unknown" in relevant analytical outputs.
- Preserve missing descriptive attributes rather than replacing them with fabricated values.
- Flag products with missing or zero weights for further investigation.
- Avoid excluding products solely because of missing metadata.

**Conclusion:**

Missing product metadata is concentrated in 610 product records. This pattern has been documented and will be considered when interpreting product category analyses.

### 2.17 Sellers Table — Data Quality Assessment

**Objective:** Examine seller information for missing values, duplicate identifiers, and geographical data quality issues.

**Importance:** Seller data quality affects seller performance analysis, geographical reporting, and logistics-related insights.

**Methodology:** Inspect the table schema before validating seller identifiers and location attributes.

In [ ]:
# Inspect sellers table schema

cur.execute("DESCRIBE sellers")

data = cur.fetchall()

df_sellers_schema = pd.DataFrame(
    data,
    columns=[
        "Column",
        "Data Type",
        "Nullable",
        "Key",
        "Default",
        "Extra"
    ]
)

display(df_sellers_schema)

,Column,Data Type,Nullable,Key,Default,Extra
0,seller_id,text,YES,,None,
1,seller_zip_code_prefix,bigint,YES,,None,
2,seller_city,text,YES,,None,
3,seller_state,text,YES,,None,


### 2.18 Sellers Table — Missing Values and Duplicate Analysis

**Objective:** Validate seller identifiers and geographical attributes.

**Importance:** Accurate seller information is necessary for reliable seller rankings, geographical segmentation, and regional performance analysis.

**Methodology:** SQL aggregation functions are used to identify missing values, duplicate seller identifiers, and blank seller IDs.

In [ ]:
# Sellers Table: Data Quality Validation

query = """
SELECT
    COUNT(*) AS total_sellers,

    COUNT(DISTINCT seller_id) AS unique_sellers,

    COUNT(*) - COUNT(DISTINCT seller_id)
        AS duplicate_seller_ids,

    SUM(seller_id IS NULL) AS missing_seller_ids,

    SUM(seller_zip_code_prefix IS NULL)
        AS missing_zip_codes,

    SUM(seller_city IS NULL) AS missing_cities,

    SUM(seller_state IS NULL) AS missing_states,

    SUM(
        CASE
            WHEN TRIM(seller_id) = '' THEN 1
            ELSE 0
        END
    ) AS blank_seller_ids

FROM sellers;
"""

cur.execute(query)

data = cur.fetchall()

df_sellers_quality = pd.DataFrame(
    data,
    columns=[
        "Total Sellers",
        "Unique Sellers",
        "Duplicate Seller IDs",
        "Missing Seller IDs",
        "Missing ZIP Codes",
        "Missing Cities",
        "Missing States",
        "Blank Seller IDs"
    ]
)

display(
    df_sellers_quality.T.rename(columns={0: "Count"})
)

,Count
Total Sellers,3095
Unique Sellers,3095
Duplicate Seller IDs,0
Missing Seller IDs,0
Missing ZIP Codes,0
Missing Cities,0
Missing States,0
Blank Seller IDs,0


**Results:**

- Total Sellers: 3,095
- Unique Seller IDs: 3,095
- Duplicate Seller IDs: 0
- Missing Values: 0
- Blank Seller IDs: 0

**Data Quality Findings:**

- All seller identifiers are unique.
- No SQL NULL values were detected in the examined seller attributes.
- No blank seller identifiers were found.
- Seller geographical information is complete in terms of SQL NULL values.

**Conclusion:**

The sellers table passed the initial missing-value and identifier uniqueness checks.

No seller records were modified or removed.

### 2.19 Geolocation Table — Data Quality Assessment

**Objective:** Examine geographical data for missing values, duplicate records, and invalid coordinates.

**Importance:** Accurate geographical information supports location-based customer analysis, delivery planning, and regional insights.

In [ ]:
# Inspect geolocation table schema

cur.execute("DESCRIBE geolocation")

data = cur.fetchall()

df_geo_schema = pd.DataFrame(
    data,
    columns=[
        "Column",
        "Data Type",
        "Nullable",
        "Key",
        "Default",
        "Extra"
    ]
)

display(df_geo_schema)

,Column,Data Type,Nullable,Key,Default,Extra
0,geolocation_zip_code_prefix,bigint,YES,,None,
1,geolocation_lat,double,YES,,None,
2,geolocation_lng,double,YES,,None,
3,geolocation_city,text,YES,,None,
4,geolocation_state,text,YES,,None,


### 2.20 Geolocation Table — Missing Values and Coordinate Validation

**Objective:** Assess the completeness and validity of geographical information.

**Importance:** Reliable geographical data supports location-based analysis, delivery planning, and regional reporting.

**Methodology:** SQL aggregation functions are used to identify missing geographical attributes, invalid latitude and longitude values, and the number of distinct postal code prefixes.

**Note:** Multiple records for the same postal code prefix are expected and are not automatically treated as duplicates.

In [ ]:
# Geolocation Table: Data Quality Validation

query = """
SELECT
    COUNT(*) AS total_records,

    COUNT(DISTINCT geolocation_zip_code_prefix)
        AS unique_zip_codes,

    SUM(geolocation_zip_code_prefix IS NULL)
        AS missing_zip_codes,

    SUM(geolocation_lat IS NULL)
        AS missing_latitudes,

    SUM(geolocation_lng IS NULL)
        AS missing_longitudes,

    SUM(geolocation_city IS NULL)
        AS missing_cities,

    SUM(geolocation_state IS NULL)
        AS missing_states,

    SUM(
        geolocation_lat < -90
        OR geolocation_lat > 90
    ) AS invalid_latitudes,

    SUM(
        geolocation_lng < -180
        OR geolocation_lng > 180
    ) AS invalid_longitudes,

    ROUND(MIN(geolocation_lat), 4)
        AS minimum_latitude,

    ROUND(MAX(geolocation_lat), 4)
        AS maximum_latitude,

    ROUND(MIN(geolocation_lng), 4)
        AS minimum_longitude,

    ROUND(MAX(geolocation_lng), 4)
        AS maximum_longitude

FROM geolocation;
"""

cur.execute(query)

data = cur.fetchall()

df_geo_quality = pd.DataFrame(
    data,
    columns=[
        "Total Records",
        "Unique ZIP Codes",
        "Missing ZIP Codes",
        "Missing Latitudes",
        "Missing Longitudes",
        "Missing Cities",
        "Missing States",
        "Invalid Latitudes",
        "Invalid Longitudes",
        "Minimum Latitude",
        "Maximum Latitude",
        "Minimum Longitude",
        "Maximum Longitude"
    ]
)

display(
    df_geo_quality.T.rename(columns={0: "Value"})
)

,Value
Total Records,1000163
Unique ZIP Codes,19015
Missing ZIP Codes,0
Missing Latitudes,0
Missing Longitudes,0
Missing Cities,0
Missing States,0
Invalid Latitudes,0
Invalid Longitudes,0
Minimum Latitude,-36.6054


**Results:**

- Total Geolocation Records: 1,000,163
- Unique ZIP Code Prefixes: 19,015
- Missing Values: 0
- Invalid Global Latitude Values: 0
- Invalid Global Longitude Values: 0

**Data Quality Findings:**

- No SQL NULL values were detected in the geographical attributes.
- All recorded coordinates fall within globally valid latitude and longitude ranges.
- Some extreme coordinate values appear inconsistent with the expected geographical coverage of Brazil.
- Multiple coordinates per ZIP code prefix are expected and do not necessarily indicate duplicate records.

**Next Investigation:**

Identify potential geographical outliers using approximate regional boundaries.

**Data Cleaning Decision:**

Preserve original geolocation records until potentially incorrect coordinates have been investigated.

### 2.21 Investigation of Potential Geographic Outliers

**Objective:** Identify geographical coordinates that fall outside approximate Brazilian latitude and longitude screening ranges.

**Importance:** Geographical outliers may distort location-based analyses and visualizations.

**Methodology:** Apply broad geographical screening thresholds to flag records requiring further investigation.

In [ ]:
# Geolocation: Potential Geographic Outlier Detection

query = """
SELECT
    COUNT(*) AS total_records,

    SUM(
        geolocation_lat < -35
        OR geolocation_lat > 6
        OR geolocation_lng < -75
        OR geolocation_lng > -30
    ) AS potential_outliers

FROM geolocation;
"""

cur.execute(query)

data = cur.fetchone()

total_records = data[0]
potential_outliers = data[1]

outlier_percentage = (
    potential_outliers / total_records
) * 100

print(f"Total Geolocation Records: {total_records:,}")
print(f"Potential Geographic Outliers: {potential_outliers:,}")
print(f"Outlier Percentage: {outlier_percentage:.4f}%")

Total Geolocation Records: 1,000,163
Potential Geographic Outliers: 29
Outlier Percentage: 0.0029%


**Results:**

- Total Geolocation Records: 1,000,163
- Potential Geographic Outliers: 29
- Potential Outlier Percentage: 0.0029%

**Data Quality Findings:**

- Approximately 99.9971% of geolocation records fall within the selected geographical screening ranges.
- Only 29 records fall outside the approximate boundaries used for Brazil.
- These records represent a very small proportion of the dataset.

**Data Cleaning Decision:**

- Preserve all original geographical records.
- Flag the 29 potential outliers for further investigation.
- Avoid removing coordinates without confirming whether they are incorrect.

**Limitations:**

The geographical screening uses approximate latitude and longitude boundaries rather than an official country polygon. Therefore, flagged records are potential outliers rather than confirmed errors.

**Conclusion:**

The geolocation table passed initial completeness and global coordinate validation checks. A small number of potential geographical outliers were identified.

### 2.22 Database Referential Integrity Validation

**Objective:** Verify that identifiers used to connect tables reference existing records.

**Importance:** Broken relationships may lead to missing records or incorrect results when joining database tables.

**Methodology:** Compare foreign-key-like identifier columns against the corresponding parent table identifiers.

**Note:** The imported tables do not currently have enforced foreign key constraints. This validation checks the relationships at the data level without modifying the database.

In [ ]:
# Final Database Relationship Validation

def get_id_set(query):
    cur.execute(query)
    return {row[0] for row in cur.fetchall() if row[0] is not None}


# Load identifiers from parent tables
customer_ids = get_id_set(
    "SELECT customer_id FROM customers"
)

order_ids = get_id_set(
    "SELECT order_id FROM orders"
)

product_ids = get_id_set(
    "SELECT product_id FROM products"
)

seller_ids = get_id_set(
    "SELECT seller_id FROM sellers"
)


# Check child table identifiers
relationships = [
    (
        "Orders → Customers",
        "SELECT customer_id FROM orders",
        customer_ids
    ),
    (
        "Payments → Orders",
        "SELECT order_id FROM payments",
        order_ids
    ),
    (
        "Order Items → Orders",
        "SELECT order_id FROM order_items",
        order_ids
    ),
    (
        "Order Items → Products",
        "SELECT product_id FROM order_items",
        product_ids
    ),
    (
        "Order Items → Sellers",
        "SELECT seller_id FROM order_items",
        seller_ids
    )
]


results = []

for relationship_name, query, valid_ids in relationships:

    cur.execute(query)
    child_ids = [row[0] for row in cur.fetchall()]

    unmatched = sum(
        1 for value in child_ids
        if value not in valid_ids
    )

    results.append({
        "Relationship": relationship_name,
        "Total Child Records": len(child_ids),
        "Unmatched References": unmatched
    })


df_relationships = pd.DataFrame(results)

display(df_relationships)

,Relationship,Total Child Records,Unmatched References
0,Orders → Customers,99441,0
1,Payments → Orders,103886,0
2,Order Items → Orders,112650,0
3,Order Items → Products,112650,0
4,Order Items → Sellers,112650,0


### 2.23 Final Data Quality Assessment Summary

**Dataset Overview:**

The e-commerce database contains 1,451,627 records distributed across seven relational tables.

**Key Findings:**

1. Customer, seller, product, and order identifiers passed the uniqueness checks performed.

2. Five major relationships between customers, orders, payments, order items, products, and sellers were validated. No unmatched child references were detected.

3. Missing order delivery timestamps are primarily associated with orders that are not marked as delivered.

4. Fourteen potentially inconsistent combinations of order status and delivery timestamp were identified.

5. Nine zero-value payment records and two zero-installment payment records were identified.

6. Six hundred and ten products have missing category and descriptive metadata.

7. Twenty-nine geolocation records were flagged as potential geographical outliers using approximate screening boundaries.

8. All non-null order date values passed Pandas datetime conversion checks.

**Data Cleaning Strategy:**

- Preserve all original records in the raw MySQL database.
- Retain valid NULL values where information is unavailable or not applicable.
- Avoid replacing missing values with fabricated information.
- Represent missing product categories as "Unknown" when appropriate for reporting.
- Flag unusual payment, delivery, product, and geographical records for further investigation.
- Use appropriate datetime and numerical types in a reproducible cleaned analytical layer.
- Document analytical assumptions and limitations.

**Referential Integrity:**

All five tested relationships returned zero unmatched references.

**Final Conclusion:**

The dataset passed the initial structural, completeness, uniqueness, and referential integrity checks, although several potential data quality issues were identified.

These findings have been documented to support transparent and reproducible business analysis.

No original database records were deleted or modified during the audit.

## 3. Quality Assessment Conclusion

- All five tested child-to-parent relationships had **zero unmatched references**.
- 14 status/delivery-date combinations need further business review.
- Payment anomalies: 9 zero-valued records, 2 zero-installment records,
  and 1 delivered order without a payment record.
- 610 products lacked all four catalog metadata fields; 2 weights were missing
  and 4 recorded weights were zero.
- 29 geolocation coordinates fell outside an **approximate** Brazil screening box.
- Source raw data was **not** altered; cleaning recommendations are documented
  for a future typed analytical layer.

Passing these checks does not mean that foreign keys are enforced, data is
free from all duplicates, or every value is semantically valid.
